# Order Volume Forecasting — Temporal Ensemble Approach
### Multi-Hub Demand Prediction for a Large-Scale Fulfilment Network

**Objective:** Predict daily `OrderVolume` for 1,115 fulfilment hubs over a 42-day horizon.  
**Metric:** Root Mean Squared Logarithmic Error (RMSLE).

| Stage | Technique | Rationale |
|:---|:---|:---|
| Feature Engineering | Calendar, hub profiles, lag/rolling, YoY lags | Temporal demand driven by seasonality, trend & hub behaviour |
| Two-Stage modelling | Stage 1 predicts AppSessions; Stage 2 uses them | App engagement is the strongest leading indicator of order intent |
| Ensemble | LightGBM + XGBoost + CatBoost + Summer Specialist | Diverse learners reduce RMSLE variance |
| Blending | Harmonic mean + optimised weights | Correct aggregator for strictly positive RMSLE targets |
| Post-processing | YoY anchoring + log-bias correction | Short-horizon calibration via same-period-last-year anchor |

## 0. Environment Setup, Imports & Runtime Configuration

In [ ]:
import subprocess, sys, os, time
GLOBAL_T0 = time.time()

def elapsed():
    return f'[{(time.time()-GLOBAL_T0)/60:.1f} min elapsed]'

for pkg in ['lightgbm', 'xgboost', 'catboost', 'seaborn', 'scipy']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, shutil
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# ── GPU detection ─────────────────────────────────────────────────────────────
USE_GPU = False
try:
    import torch
    USE_GPU = torch.cuda.is_available()
except Exception:
    try:
        import subprocess as _sp
        USE_GPU = _sp.run(['nvidia-smi'], capture_output=True).returncode == 0
    except Exception:
        pass

print(f'GPU available: {USE_GPU}')

# ── Adaptive parameters: GPU = full quality, CPU = 30-minute safe mode ────────
if USE_GPU:
    N_EST     = 5000     # full convergence on GPU (~2 min/model)
    LR        = 0.02     # slow learning + deep convergence
    CB_DEPTH  = 8
    CB_ITERS  = 5000
    SUM_EST   = 3000
    print('Mode: GPU (full quality, ~15-20 min total)')
else:
    N_EST     = 2000     # CPU-safe: still competitive, ~25 min total
    LR        = 0.05
    CB_DEPTH  = 6
    CB_ITERS  = 2000
    SUM_EST   = 1000
    print('Mode: CPU (fast, ~25 min total)')

# ── Auto-detect DATA_PATH (Kaggle or local) ───────────────────────────────────
_KAGGLE_PATH = '/kaggle/input/competitions/ch-27-celebal-technologies-vit-vellore/'
_LOCAL_PATH  = './Dataset/'
if os.path.exists(_KAGGLE_PATH + 'orders_train.csv'):
    DATA_PATH = _KAGGLE_PATH
elif os.path.exists(_LOCAL_PATH + 'orders_train.csv'):
    DATA_PATH = _LOCAL_PATH
else:
    raise FileNotFoundError('Dataset not found. Expected: /kaggle/input/... or ./Dataset/')
print(f'Data path: {DATA_PATH}')

pd.set_option('display.max_columns', 50)
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',   'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',      'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',       'grid.color': '#21262d',
})
PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']
SEED = 42
np.random.seed(SEED)
print(f'Setup complete. {elapsed()}')

## 1. Load & Inspect Data

In [ ]:
train  = pd.read_csv(DATA_PATH + 'orders_train.csv')
test   = pd.read_csv(DATA_PATH + 'orders_test.csv')
meta   = pd.read_csv(DATA_PATH + 'hub_metadata.csv')
sample = pd.read_csv(DATA_PATH + 'sample_submission.csv')

train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])

print(f'Train : {train.shape[0]:>7,} rows | {train["Date"].min().date()} -> {train["Date"].max().date()}')
print(f'Test  : {test.shape[0]:>7,} rows | {test["Date"].min().date()} -> {test["Date"].max().date()}')
print(f'Meta  : {meta.shape[0]:>7,} rows | {meta.shape[1]} columns')
print(f'Unique hubs: {train["HubID"].nunique()}')
print(f'\nOrderVolume (open days):')
print(train[train['IsOpen']==1]['OrderVolume'].describe().round(1))
print(elapsed())
train.head(3)

## 2. Feature Engineering

Demand forecasting for fulfilment hubs is a multi-scale problem. We engineer five categories:

1. **Calendar & hub metadata** — weekday, month, seasonality, hub attributes
2. **Hub-level demand profiles** — per-hub promotion lift, holiday elasticity, open-day patterns
3. **Temporal trend** — per-hub linear sales trend from the most recent 365 days
4. **Lag & rolling features** — short-term memory (7–56 days), YoY anchors (364/365/371 days)
5. **Target encodings** — hub × weekday, hub × month OOF historical means

### 2.1 Calendar + Metadata Features

Standard temporal decomposition with cyclical sin/cos encodings to preserve
calendar continuity (e.g., week 52 → week 1 is a small step, not a jump of 51).

In [ ]:
LOYALTY_MONTHS = {
    'a': [1,4,7,10], 'b': [2,5,8,11], 'c': [3,6,9,12],
    'd': [1,3,5,7,9,11], 'e': [2,4,6,8,10,12],
}

def engineer_base(df, meta_df):
    df = df.merge(meta_df, on='HubID', how='left')
    df['Year']       = df['Date'].dt.year
    df['Month']      = df['Date'].dt.month
    df['Week']       = df['Date'].dt.isocalendar().week.astype(int)
    df['DayOfYear']  = df['Date'].dt.dayofyear
    df['DayOfMonth'] = df['Date'].dt.day
    df['Quarter']    = df['Date'].dt.quarter
    df['IsSummer']   = df['Month'].isin([6,7,8]).astype(np.int8)

    df['Month_sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Week_sin']    = np.sin(2*np.pi*df['Week']/52)
    df['Week_cos']    = np.cos(2*np.pi*df['Week']/52)
    df['Weekday_sin'] = np.sin(2*np.pi*df['Weekday']/7)
    df['Weekday_cos'] = np.cos(2*np.pi*df['Weekday']/7)

    df['HasCompetitor']        = df['CompetitorDistance'].notna().astype(int)
    df['CompetitorDist_log']   = np.log1p(df['CompetitorDistance'].fillna(0))
    df['HasLoyalty']           = df['LoyaltyProgramSinceYear'].notna().astype(int)
    df['LoyaltyInterval']      = df['LoyaltyProgramInterval'].map(
        {'a':0,'b':1,'c':2,'d':3,'e':4}).fillna(-1)
    df['CompetitorAge_months'] = (
        (df['Year']  - df['CompetitorOpenSinceYear'].fillna(df['Year'])) * 12 +
        (df['Month'] - df['CompetitorOpenSinceMonth'].fillna(df['Month']))
    ).clip(lower=0)
    df['LoyaltyAge_weeks'] = (
        (df['Year'] - df['LoyaltyProgramSinceYear'].fillna(df['Year'])) * 52 +
        (df['Week'] - df['LoyaltyProgramSinceWeek'].fillna(df['Week']))
    ).clip(lower=0)

    def loyalty_active(row):
        if row['HasLoyalty'] == 0: return 0
        iv = row['LoyaltyProgramInterval']
        return 1 if (iv in LOYALTY_MONTHS and row['Month'] in LOYALTY_MONTHS[iv]) else 0
    df['LoyaltyActiveMonth'] = df.apply(loyalty_active, axis=1).astype(np.int8)

    df['IsWeekend']         = (df['Weekday'] >= 6).astype(np.int8)
    df['AnyHoliday']        = (df['RegionalHoliday'] > 0).astype(int)
    df['Promo_x_Weekend']   = df['PromoActive'] * df['IsWeekend']
    df['Promo_x_Holiday']   = df['PromoActive'] * df['AnyHoliday']
    df['Promo_x_School']    = df['PromoActive'] * df['SchoolClosureFlag']
    df['Holiday_x_Weekend'] = df['AnyHoliday'] * df['IsWeekend']
    df['Loyalty_x_Promo']   = df['LoyaltyActiveMonth'] * df['PromoActive']
    return df

print('Engineering base features...')
train_fe = engineer_base(train, meta)
test_fe  = engineer_base(test,  meta)
print(f'Done. Train: {train_fe.shape}  Test: {test_fe.shape}  {elapsed()}')

### 2.2 Hub-Level Demand Profile Features

Each hub has a unique demand elasticity. A hub in a commercial district may see 40% uplift
during promotions; a residential hub may see only 5%. We compute static per-hub ratios
(promotion lift, holiday response, Saturday boost, open-rate) from the full training history.

In [ ]:
def add_hub_ratio_features(train_df, test_df):
    open_df = train_df[train_df['IsOpen'] == 1].copy()
    overall = open_df.groupby('HubID')['OrderVolume'].mean().rename('hub_mean_vol')
    hub_stats = pd.DataFrame({'hub_mean_vol': overall})

    promo_mean = open_df[open_df['PromoActive']==1].groupby('HubID')['OrderVolume'].mean()
    hub_stats['promo_sales_ratio']    = (promo_mean / overall).fillna(1.0)
    hol_mean   = open_df[open_df['AnyHoliday']==1].groupby('HubID')['OrderVolume'].mean()
    hub_stats['holiday_sales_ratio']  = (hol_mean / overall).fillna(1.0)
    sat_mean   = open_df[open_df['Weekday'].isin([6])].groupby('HubID')['OrderVolume'].mean()
    hub_stats['saturday_sales_ratio'] = (sat_mean / overall).fillna(1.0)
    hub_stats['school_hol_pct'] = open_df.groupby('HubID')['SchoolClosureFlag'].mean()
    total_days = train_df.groupby('HubID')['Date'].count()
    open_days  = open_df.groupby('HubID')['Date'].count()
    hub_stats['open_rate'] = (open_days / total_days).fillna(0)

    for col in ['promo_sales_ratio','holiday_sales_ratio','saturday_sales_ratio']:
        hub_stats[col] = hub_stats[col].clip(0.5, 3.0)
    hub_stats = hub_stats.reset_index()
    train_df = train_df.merge(hub_stats, on='HubID', how='left')
    test_df  = test_df.merge(hub_stats, on='HubID', how='left')
    for col in hub_stats.columns:
        if col == 'HubID': continue
        train_df[col] = train_df[col].fillna(train_df[col].median())
        test_df[col]  = test_df[col].fillna(train_df[col].median())
    return train_df, test_df

print('Computing per-hub demand profiles...')
train_fe, test_fe = add_hub_ratio_features(train_fe, test_fe)
print(f'Done. {elapsed()}')

### 2.3 Per-Hub Sales Trend (Ridge Regression)

Hubs follow long-term growth or decline trajectories driven by neighbourhood development,
competitor entry, and customer base changes. We fit a per-hub Ridge regression on the
most recent 365 days to extract the hub's trend slope and project it into the test period.

In [ ]:
def add_hub_trend_features(train_df, test_df):
    cutoff         = train_df['Date'].max()
    lookback_start = cutoff - pd.Timedelta(days=365)
    recent = train_df[
        (train_df['IsOpen'] == 1) & (train_df['Date'] >= lookback_start)
    ].copy()
    recent['day_num'] = (recent['Date'] - lookback_start).dt.days

    hub_slopes, hub_intercepts = {}, {}
    for hub_id, grp in recent.groupby('HubID'):
        if len(grp) < 14:
            hub_slopes[hub_id]     = 0.0
            hub_intercepts[hub_id] = grp['OrderVolume'].mean()
            continue
        X_r = grp[['day_num','Weekday','PromoActive']].values.astype(float)
        y_r = np.log1p(grp['OrderVolume'].values)
        X_s = StandardScaler().fit_transform(X_r)
        r   = Ridge(alpha=1.0).fit(X_s, y_r)
        hub_slopes[hub_id]     = r.coef_[0]
        hub_intercepts[hub_id] = r.intercept_

    train_df = train_df.copy(); test_df = test_df.copy()
    train_df['hub_trend_slope']     = train_df['HubID'].map(hub_slopes).fillna(0)
    train_df['hub_trend_intercept'] = train_df['HubID'].map(hub_intercepts).fillna(
        train_df['HubID'].map(recent.groupby('HubID')['OrderVolume'].mean()))
    test_df['hub_trend_slope']      = test_df['HubID'].map(hub_slopes).fillna(0)
    test_df['hub_trend_intercept']  = test_df['HubID'].map(hub_intercepts).fillna(
        test_df['HubID'].map(recent.groupby('HubID')['OrderVolume'].mean()))
    train_df['days_from_cutoff'] = (train_df['Date'] - cutoff).dt.days
    test_df['days_from_cutoff']  = (test_df['Date']  - cutoff).dt.days
    train_df['hub_trend_proj'] = np.expm1(
        (train_df['hub_trend_slope'] * train_df['days_from_cutoff'] +
         train_df['hub_trend_intercept']).clip(-5, 12))
    test_df['hub_trend_proj']  = np.expm1(
        (test_df['hub_trend_slope']  * test_df['days_from_cutoff']  +
         test_df['hub_trend_intercept']).clip(-5, 12))
    print(f'Slope range: [{min(hub_slopes.values()):.4f}, {max(hub_slopes.values()):.4f}]  '
          f'Growing hubs: {sum(1 for v in hub_slopes.values() if v>0)}')
    return train_df, test_df

print('Fitting per-hub trend models...')
train_fe, test_fe = add_hub_trend_features(train_fe, test_fe)
print(f'Done. {elapsed()}')

### 2.4 Promotional Timing Features

Demand builds before a promotion and decays after it. Day-counters since/until
the nearest promotion capture this temporal response curve.

In [ ]:
def add_promo_distance(train_df, test_df):
    test_tmp = test_df.copy()
    if 'OrderVolume' not in test_tmp.columns: test_tmp['OrderVolume'] = np.nan
    full = pd.concat([
        train_df[['HubID','Date','PromoActive']],
        test_tmp[['HubID','Date','PromoActive']]
    ], ignore_index=True).sort_values(['HubID','Date']).reset_index(drop=True)

    results = []
    for hub_id, grp in full.groupby('HubID'):
        grp  = grp.sort_values('Date').reset_index(drop=True)
        dates = grp['Date'].values; promo = grp['PromoActive'].values; n = len(grp)
        days_since = np.full(n, -1, dtype=np.float32)
        days_until = np.full(n, -1, dtype=np.float32)
        last_p = None
        for i in range(n):
            if promo[i] == 1: last_p = dates[i]
            if last_p is not None: days_since[i] = (dates[i]-last_p)/np.timedelta64(1,'D')
        next_p = None
        for i in range(n-1,-1,-1):
            if promo[i] == 1: next_p = dates[i]
            if next_p is not None: days_until[i] = (next_p-dates[i])/np.timedelta64(1,'D')
        grp['DaysSinceLastPromo'] = days_since
        grp['DaysUntilNextPromo'] = days_until
        results.append(grp[['HubID','Date','DaysSinceLastPromo','DaysUntilNextPromo']])

    promo_dist = pd.concat(results, ignore_index=True)
    cutoff = train_df['Date'].max()
    train_df = train_df.merge(promo_dist[promo_dist['Date']<=cutoff], on=['HubID','Date'], how='left')
    test_df  = test_df.merge(promo_dist[promo_dist['Date']>cutoff],  on=['HubID','Date'], how='left')
    return train_df, test_df

print('Computing promotional timing features...')
train_fe, test_fe = add_promo_distance(train_fe, test_fe)
print(f'Done. {elapsed()}')

### 2.5 Reopening Demand Surge (Pent-Up Demand)

In [ ]:
def add_open_after_closed(train_df, test_df):
    test_tmp = test_df.copy()
    if 'OrderVolume' not in test_tmp.columns: test_tmp['OrderVolume'] = np.nan
    full = pd.concat([
        train_df[['HubID','Date','IsOpen']],
        test_tmp[['HubID','Date','IsOpen']]
    ], ignore_index=True).sort_values(['HubID','Date']).reset_index(drop=True)
    full['IsOpen_Lag1']     = full.groupby('HubID')['IsOpen'].shift(1)
    full['OpenAfterClosed'] = ((full['IsOpen']==1) & (full['IsOpen_Lag1']==0)).astype(np.int8)
    full['closed_streak']   = (full['IsOpen']==0).astype(int)
    full['ClosedDaysStreak'] = full.groupby('HubID')['closed_streak'].transform(
        lambda x: x.groupby((x!=x.shift()).cumsum()).transform('cumsum')
    ).shift(1).fillna(0)
    cols = ['HubID','Date','IsOpen_Lag1','OpenAfterClosed','ClosedDaysStreak']
    cutoff = train_df['Date'].max()
    train_df = train_df.merge(full[full['Date']<=cutoff][cols], on=['HubID','Date'], how='left')
    test_df  = test_df.merge(full[full['Date']>cutoff][cols],  on=['HubID','Date'], how='left')
    for col in ['IsOpen_Lag1','OpenAfterClosed','ClosedDaysStreak']:
        train_df[col] = train_df[col].fillna(0)
        test_df[col]  = test_df[col].fillna(0)
    return train_df, test_df

print('Computing reopening demand features...')
train_fe, test_fe = add_open_after_closed(train_fe, test_fe)
reopened = train_fe[train_fe['OpenAfterClosed']==1]['OrderVolume'].mean()
normal   = train_fe[(train_fe['IsOpen']==1)&(train_fe['OpenAfterClosed']==0)]['OrderVolume'].mean()
print(f'Reopen-day demand uplift: {reopened/normal:.3f}x  {elapsed()}')

### 2.6 Lag & Rolling Features — With YoY Imputation

Year-over-year lags (364/365/371 days) are the strongest predictors — they capture
the same seasonal regime, weekday, and hub maturity from the prior year.

**Critical design:** Short-term lags (vol_lag_7, vol_lag_14 etc.) for test rows beyond
day 7 would look back into the unknown test period (NaN → zero after fillna).
The model interprets zero as "hub was recently closed" and over-predicts pent-up demand.

**Fix applied here:** NaN short-term lags are replaced with the corresponding YoY lag
(always observable from training data) before the dataframe is split and merged.

In [ ]:
def add_lag_rolling_features(train_df, test_df):
    test_tmp = test_df.copy()
    if 'OrderVolume' not in test_tmp.columns: test_tmp['OrderVolume'] = np.nan
    if 'AppSessions'  not in test_tmp.columns: test_tmp['AppSessions']  = np.nan

    full = pd.concat([
        train_df[['HubID','Date','OrderVolume','AppSessions','IsOpen']],
        test_tmp[['HubID','Date','OrderVolume','AppSessions','IsOpen']]
    ], ignore_index=True).sort_values(['HubID','Date']).reset_index(drop=True)

    for lag in [7, 14, 21, 28, 42, 56]:
        full[f'vol_lag_{lag}'] = full.groupby('HubID')['OrderVolume'].shift(lag)
    for lag in [91, 182, 364, 365, 371, 728]:
        full[f'vol_lag_{lag}'] = full.groupby('HubID')['OrderVolume'].shift(lag)
    full['vol_yoy_ratio_364'] = (full['vol_lag_7']+1) / (full['vol_lag_371']+1)
    full['vol_yoy_ratio_728'] = (full['vol_lag_7']+1) / (full['vol_lag_728']+1)

    for w in [7, 14, 28, 56, 91]:
        full[f'vol_roll{w}_mean'] = full.groupby('HubID')['OrderVolume'].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        full[f'vol_roll{w}_max']  = full.groupby('HubID')['OrderVolume'].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).max())
    full['vol_expand_mean'] = full.groupby('HubID')['OrderVolume'].transform(
        lambda x: x.shift(1).expanding().mean())
    full['vol_roll7_dev'] = full['vol_roll7_mean'] - full['vol_expand_mean']

    for lag in [7,14,28]:
        full[f'appsess_lag_{lag}'] = full.groupby('HubID')['AppSessions'].shift(lag)
    for w in [7,14,28]:
        full[f'appsess_roll{w}_mean'] = full.groupby('HubID')['AppSessions'].transform(
            lambda x: x.shift(1).rolling(w, min_periods=1).mean())

    # ── YoY imputation for test rows ─────────────────────────────────────────
    # Short-term lags that fall inside the test window are replaced with same-
    # day-last-year values so test features stay in the training distribution.
    cutoff    = train_df['Date'].max()
    test_mask = full['Date'] > cutoff
    yoy_proxy = {
        'vol_lag_7'      : 'vol_lag_364',
        'vol_lag_14'     : 'vol_lag_364',
        'vol_lag_21'     : 'vol_lag_365',
        'vol_lag_28'     : 'vol_lag_371',
        'vol_roll7_mean' : 'vol_lag_364',
        'vol_roll14_mean': 'vol_lag_365',
        'vol_roll28_mean': 'vol_lag_371',
    }
    replaced = 0
    for short, yoy in yoy_proxy.items():
        if short in full.columns and yoy in full.columns:
            na_rows = test_mask & full[short].isna() & full[yoy].notna()
            full.loc[na_rows, short] = full.loc[na_rows, yoy]
            replaced += na_rows.sum()
    full['vol_roll7_dev'] = full['vol_roll7_mean'] - full['vol_expand_mean']
    print(f'  YoY imputation: {replaced:,} test lag values replaced with same-day-last-year')

    lag_cols = [c for c in full.columns
                if c.startswith(('vol_lag','vol_roll','vol_expand','vol_yoy','appsess'))]
    return (
        full[full['Date']<=cutoff][['HubID','Date']+lag_cols].copy(),
        full[full['Date']>cutoff][['HubID','Date']+lag_cols].copy(),
        lag_cols
    )

print('Computing lag & rolling features (with YoY imputation for test)...')
train_lags, test_lags, lag_cols = add_lag_rolling_features(train_fe, test_fe)
train_fe = train_fe.merge(train_lags, on=['HubID','Date'], how='left')
test_fe  = test_fe.merge(test_lags,  on=['HubID','Date'], how='left')
print(f'Lag features: {len(lag_cols)}  {elapsed()}')

### 2.7 OOF Target Encodings

Hub-level and hub × weekday mean encodings — computed out-of-fold to prevent leakage.

In [ ]:
def oof_hub_target_encode(train_df, test_df, target='OrderVolume', n_splits=5):
    train_sorted = train_df.sort_values('Date').copy()
    oof_vals = np.zeros(len(train_sorted))
    tscv_enc = TimeSeriesSplit(n_splits=n_splits)
    for _, (tr_idx, val_idx) in enumerate(tscv_enc.split(train_sorted)):
        hub_means   = train_sorted.iloc[tr_idx].groupby('HubID')[target].mean()
        global_mean = train_sorted.iloc[tr_idx][target].mean()
        oof_vals[val_idx] = train_sorted.iloc[val_idx]['HubID'].map(hub_means).fillna(global_mean).values
    train_df = train_df.copy()
    train_df['hub_oof_mean_vol'] = oof_vals
    hub_means_full = train_df.groupby('HubID')[target].mean()
    test_df = test_df.copy()
    test_df['hub_oof_mean_vol'] = test_df['HubID'].map(hub_means_full).fillna(train_df[target].mean())
    return train_df, test_df

def hub_weekday_encode(train_df, test_df, target='OrderVolume'):
    open_train = train_df[train_df['IsOpen']==1]
    hw = open_train.groupby(['HubID','Weekday'])[target].mean().reset_index()
    hw.columns = ['HubID','Weekday','hub_weekday_mean_vol']
    hub_mean = open_train.groupby('HubID')[target].mean()
    train_df = train_df.merge(hw, on=['HubID','Weekday'], how='left')
    test_df  = test_df.merge(hw, on=['HubID','Weekday'], how='left')
    train_df['hub_weekday_mean_vol'] = train_df['hub_weekday_mean_vol'].fillna(train_df['HubID'].map(hub_mean))
    test_df['hub_weekday_mean_vol']  = test_df['hub_weekday_mean_vol'].fillna(test_df['HubID'].map(hub_mean))
    return train_df, test_df

def hub_month_encode(train_df, test_df, target='OrderVolume'):
    open_train = train_df[train_df['IsOpen']==1].copy()
    hm = open_train.groupby(['HubID','Month'])[target].mean().reset_index()
    hm.columns = ['HubID','Month','hub_month_mean_vol']
    hub_mean = open_train.groupby('HubID')[target].mean()
    train_df = train_df.merge(hm, on=['HubID','Month'], how='left')
    test_df  = test_df.merge(hm, on=['HubID','Month'], how='left')
    train_df['hub_month_mean_vol'] = train_df['hub_month_mean_vol'].fillna(train_df['HubID'].map(hub_mean))
    test_df['hub_month_mean_vol']  = test_df['hub_month_mean_vol'].fillna(test_df['HubID'].map(hub_mean))
    return train_df, test_df

print('Computing OOF target encodings...')
train_fe, test_fe = oof_hub_target_encode(train_fe, test_fe)
train_fe, test_fe = hub_weekday_encode(train_fe, test_fe)
train_fe, test_fe = hub_month_encode(train_fe, test_fe)
print(f'Done. {elapsed()}')

### 2.8 Two-Stage: Predict App Engagement for the Forecast Period

App sessions are not available for test dates. A Stage 1 model predicts them
from observable calendar + hub features, then the predicted sessions are used as
a feature in the main Stage 2 models.

In [ ]:
def stage1_predict_appsessions(train_df, test_df):
    drop_s1 = ['Date','OrderVolume','AppSessions',
                'CompetitorOpenSinceMonth','CompetitorOpenSinceYear',
                'LoyaltyProgramSinceWeek','LoyaltyProgramSinceYear',
                'LoyaltyProgramInterval','CompetitorDistance','Id','closed_streak']
    train_s1 = train_df[(train_df['IsOpen']==1) & train_df['AppSessions'].notna()].copy()
    feat_cols = [c for c in train_s1.columns
                 if c not in drop_s1 and c != 'HubID'
                 and train_s1[c].dtype in [np.float64,np.float32,np.int64,np.int32,np.int8,bool]]
    key_lag  = [f for f in ['appsess_lag_7','appsess_lag_14'] if f in feat_cols]
    train_s1_clean = train_s1.dropna(subset=key_lag).sort_values('Date').reset_index(drop=True)
    X_s1 = train_s1_clean[feat_cols].fillna(0)
    y_s1 = train_s1_clean['AppSessions'].values
    s1_params = {'device': 'gpu'} if USE_GPU else {}
    lgb_s1 = lgb.LGBMRegressor(
        objective='regression', n_estimators=1000, learning_rate=0.05,
        num_leaves=63, feature_fraction=0.6, bagging_fraction=0.8,
        bagging_freq=1, n_jobs=-1, verbosity=-1, random_state=SEED, **s1_params)
    lgb_s1.fit(X_s1, y_s1)
    test_open = test_df[test_df['IsOpen']==1].copy()
    test_df   = test_df.copy()
    test_df['pred_AppSessions'] = 0.0
    test_df.loc[test_df['IsOpen']==1,'pred_AppSessions'] = np.maximum(
        lgb_s1.predict(test_open[feat_cols].fillna(0)), 0)
    train_df  = train_df.copy()
    train_df['pred_AppSessions'] = train_df['AppSessions'].fillna(0)
    print(f'Stage 1 done. Test pred_AppSessions mean={test_df["pred_AppSessions"].mean():.1f}  '
          f'(train mean: {train_df["AppSessions"].mean():.1f})')
    return train_df, test_df

print('Stage 1: predicting app engagement...')
train_fe, test_fe = stage1_predict_appsessions(train_fe, test_fe)
print(f'{elapsed()}')

### 2.9 Final Feature Matrix Assembly

In [ ]:
DROP_COLS = [
    'Date','OrderVolume','AppSessions',
    'CompetitorOpenSinceMonth','CompetitorOpenSinceYear',
    'LoyaltyProgramSinceWeek','LoyaltyProgramSinceYear',
    'LoyaltyProgramInterval','CompetitorDistance','Id','closed_streak',
]
train_model  = train_fe[train_fe['IsOpen']==1].copy()
FEATURE_COLS = [c for c in train_model.columns
                if c not in DROP_COLS and c != 'HubID'
                and train_model[c].dtype in [np.float64,np.float32,np.int64,np.int32,np.int8,bool]]
if 'Month' not in FEATURE_COLS and 'Month' in train_model.columns:
    FEATURE_COLS.append('Month')

print(f'Total features: {len(FEATURE_COLS)}')
key_lags = [f'vol_lag_{d}' for d in [7,14,21] if f'vol_lag_{d}' in train_model.columns]
train_model_clean = train_model.dropna(subset=key_lags).sort_values('Date').reset_index(drop=True)
print(f'Training rows (open + lags available): {len(train_model_clean):,}')

X      = train_model_clean[FEATURE_COLS].fillna(0)
y      = np.log1p(train_model_clean['OrderVolume'])
y_raw  = train_model_clean['OrderVolume'].values
X_test = test_fe[FEATURE_COLS].fillna(0)

nan_tr = X.isnull().sum();      nan_tr = nan_tr[nan_tr>0]
nan_te = X_test.isnull().sum(); nan_te = nan_te[nan_te>0]
print(f'Train NaN: {"None" if nan_tr.empty else nan_tr.to_dict()}')
print(f'Test  NaN: {"None" if nan_te.empty else nan_te.to_dict()}')
print(f'X: {X.shape}  X_test: {X_test.shape}')
vol7_nonzero_pct = (X_test['vol_lag_7']!=0).mean()*100
print(f'vol_lag_7 non-zero in test: {vol7_nonzero_pct:.1f}%  (should be >80% after YoY imputation)')
print(f'Feature matrix ready. {elapsed()}')

## 3. Validation Strategy & Metric

`RMSLE = sqrt(mean((log(pred+1) - log(actual+1))^2))`

We use `TimeSeriesSplit(n_splits=5)` — each validation fold always comes after its
training fold, mimicking the production forecasting scenario.

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

covered_mask = np.zeros(len(X), dtype=bool)
for _, val_idx in tscv.split(X): covered_mask[val_idx] = True

print(f'Metric: RMSLE')
print(f'Naive mean baseline: {rmsle(y_raw, np.full_like(y_raw, y_raw.mean(), dtype=float)):.5f}')
print(f'Covered rows: {covered_mask.sum():,} / {len(X):,}')
print(f'{elapsed()}')

## 4. LightGBM

`feature_fraction=0.30`: With 60+ correlated temporal features, aggressive column subsampling
forces each tree to use diverse feature subsets, preventing dominant features (AppSessions lags)
from masking weaker but complementary signals. This substantially improves generalisation.

In [ ]:
LGB_PARAMS = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'learning_rate'    : LR,
    'num_leaves'       : 127,
    'min_child_samples': 40,
    'feature_fraction' : 0.30,
    'bagging_fraction' : 0.80,
    'bagging_freq'     : 1,
    'reg_alpha'        : 0.05,
    'reg_lambda'       : 1.0,
    'n_estimators'     : N_EST,
    'random_state'     : SEED,
    'verbosity'        : -1,
    'n_jobs'           : -1,
}
if USE_GPU: LGB_PARAMS['device'] = 'gpu'

oof_lgb, pred_lgb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_lgb = []
lgb_fi        = pd.Series(0.0, index=FEATURE_COLS)

print(f'Training LightGBM (n_estimators={N_EST}, lr={LR})...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(500)])
    val_pred      = np.maximum(np.expm1(model.predict(X.iloc[val_idx])), 0)
    oof_lgb[val_idx] = val_pred
    score         = rmsle(y_raw[val_idx], val_pred)
    cv_scores_lgb.append(score)
    pred_lgb     += np.maximum(np.expm1(model.predict(X_test)), 0) / N_SPLITS
    lgb_fi       += pd.Series(model.feature_importances_, index=FEATURE_COLS) / N_SPLITS
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration_}')

lgb_oof = np.mean(cv_scores_lgb)
print(f'\nLightGBM OOF RMSLE: {lgb_oof:.5f} +/- {np.std(cv_scores_lgb):.5f}  {elapsed()}')

In [ ]:
top30 = lgb_fi.sort_values(ascending=False).head(30)
fig, ax = plt.subplots(figsize=(11, 9))
ax.barh(top30.index[::-1], top30.values[::-1], color=PALETTE[0], alpha=0.9)
ax.set_title('Top 30 Feature Importances', fontsize=13)
ax.set_xlabel('Importance')
ax.grid(True, axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

## 5. XGBoost

XGBoost's L1/L2 regularisation and depth-wise growth complement LightGBM's
leaf-wise strategy, producing different error patterns that reduce ensemble variance.

In [ ]:
XGB_PARAMS = {
    'objective'        : 'reg:squarederror',
    'learning_rate'    : LR,
    'max_depth'        : 7,
    'n_estimators'     : N_EST,
    'subsample'        : 0.80,
    'colsample_bytree' : 0.30,
    'reg_alpha'        : 0.05,
    'reg_lambda'       : 1.0,
    'min_child_weight' : 40,
    'tree_method'      : 'hist',
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbosity'        : 0,
}
if USE_GPU: XGB_PARAMS['device'] = 'cuda'

oof_xgb, pred_xgb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_xgb = []

print(f'Training XGBoost (n_estimators={N_EST}, lr={LR})...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=150, eval_metric='rmse')
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], verbose=500)
    val_pred       = np.maximum(np.expm1(model.predict(X.iloc[val_idx])), 0)
    oof_xgb[val_idx] = val_pred
    score          = rmsle(y_raw[val_idx], val_pred)
    cv_scores_xgb.append(score)
    pred_xgb      += np.maximum(np.expm1(model.predict(X_test)), 0) / N_SPLITS
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration}')

xgb_oof = np.mean(cv_scores_xgb)
print(f'\nXGBoost OOF RMSLE: {xgb_oof:.5f} +/- {np.std(cv_scores_xgb):.5f}  {elapsed()}')

## 6. CatBoost

CatBoost's symmetric oblivious trees generalise well on tabular data with mixed
feature types. Its built-in categorical handling eliminates the need for manual
encoding of hub format and assortment tier features.

In [ ]:
CAT_FEATURES = [c for c in ['HubFormat','AssortmentTier','LoyaltyProgram',
                              'IsOpen','PromoActive','RegionalHoliday','SchoolClosureFlag',
                              'Weekday','IsWeekend','AnyHoliday','LoyaltyActiveMonth',
                              'OpenAfterClosed','Quarter','IsSummer'] if c in FEATURE_COLS]

CB_PARAMS = {
    'iterations'           : CB_ITERS,
    'learning_rate'        : LR,
    'depth'                : CB_DEPTH,
    'l2_leaf_reg'          : 3,
    'random_strength'      : 1,
    'bagging_temperature'  : 1,
    'cat_features'         : CAT_FEATURES,
    'eval_metric'          : 'RMSE',
    'early_stopping_rounds': 150,
    'random_seed'          : SEED,
    'verbose'              : 500,
    'thread_count'         : -1,
}
if USE_GPU:
    CB_PARAMS['task_type'] = 'GPU'
    # rsm (column subsampling) is not supported in GPU regression mode
    print('CatBoost: GPU mode. rsm omitted (not supported in GPU regression).')
else:
    CB_PARAMS['task_type'] = 'CPU'
    CB_PARAMS['rsm']       = 0.30
    print('CatBoost: CPU mode. rsm=0.30 enabled.')

oof_cb, pred_cb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_cb    = []

print(f'Training CatBoost (iterations={CB_ITERS}, depth={CB_DEPTH})...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_tr_cb, X_val_cb = X.iloc[tr_idx].copy(), X.iloc[val_idx].copy()
    for col in CAT_FEATURES:
        X_tr_cb[col]  = X_tr_cb[col].astype(str)
        X_val_cb[col] = X_val_cb[col].astype(str)
    model = cb.CatBoostRegressor(**CB_PARAMS)
    model.fit(X_tr_cb, y.iloc[tr_idx], eval_set=(X_val_cb, y.iloc[val_idx]))
    val_pred      = np.maximum(np.expm1(model.predict(X_val_cb)), 0)
    oof_cb[val_idx] = val_pred
    score         = rmsle(y_raw[val_idx], val_pred)
    cv_scores_cb.append(score)
    X_test_cb = X_test.copy()
    for col in CAT_FEATURES: X_test_cb[col] = X_test_cb[col].astype(str)
    pred_cb += np.maximum(np.expm1(model.predict(X_test_cb)), 0) / N_SPLITS
    print(f'  Fold {fold}: RMSLE={score:.5f}')

cb_oof = np.mean(cv_scores_cb)
print(f'\nCatBoost OOF RMSLE: {cb_oof:.5f} +/- {np.std(cv_scores_cb):.5f}  {elapsed()}')

## 7. Summer Specialist Model

The evaluation window (June–July) falls entirely in summer. A specialist model trained
exclusively on May–September data across all training years captures summer-specific
patterns without dilution from winter months.

In [ ]:
summer_mask  = train_model_clean['Month'].isin([5,6,7,8,9])
X_summer     = X[summer_mask].reset_index(drop=True)
y_summer     = y[summer_mask].reset_index(drop=True)
y_raw_summer = y_raw[summer_mask]
print(f'Summer specialist rows: {len(X_summer):,}')

tscv_summer  = TimeSeriesSplit(n_splits=3)
oof_summer   = np.zeros(len(X))
pred_summer  = np.zeros(len(X_test))
cv_scores_summer = []

LGB_SUMMER = {**LGB_PARAMS, 'learning_rate': min(LR*1.5, 0.05), 'n_estimators': SUM_EST}
summer_idx = np.where(summer_mask)[0]

print('Training summer specialist...')
for fold, (tr_idx_s, val_idx_s) in enumerate(tscv_summer.split(X_summer), 1):
    model = lgb.LGBMRegressor(**LGB_SUMMER)
    model.fit(X_summer.iloc[tr_idx_s], y_summer.iloc[tr_idx_s],
              eval_set=[(X_summer.iloc[val_idx_s], y_summer.iloc[val_idx_s])],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)])
    val_pred = np.maximum(np.expm1(model.predict(X_summer.iloc[val_idx_s])), 0)
    oof_summer[summer_idx[val_idx_s]] = val_pred
    score    = rmsle(y_raw_summer[val_idx_s], val_pred)
    cv_scores_summer.append(score)
    pred_summer += np.maximum(np.expm1(model.predict(X_test)), 0) / 3
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration_}')

# Non-summer OOF rows must be filled so harmonic mean stays valid
non_summer_mask = (oof_summer == 0)
oof_summer[non_summer_mask] = oof_lgb[non_summer_mask]

summer_oof = np.mean(cv_scores_summer)
print(f'\nSummer specialist OOF RMSLE: {summer_oof:.5f}  {elapsed()}')

## 8. Harmonic Mean Ensemble

The harmonic mean is the theoretically optimal aggregator for RMSLE: it equals the geometric
mean of predictions (log-space average), matching RMSLE's log-space measurement. It pulls
predictions toward smaller values, reducing the over-prediction risk RMSLE penalises.

In [ ]:
def weighted_harmonic_mean(preds_matrix, weights):
    w          = np.array(weights); w = w / w.sum()
    safe_preds = np.maximum(preds_matrix, 1e-9)
    return w.sum() / np.sum(w[np.newaxis,:] / safe_preds, axis=1)

oof_stack  = np.column_stack([oof_lgb, oof_xgb, oof_cb, oof_summer])
test_stack = np.column_stack([pred_lgb, pred_xgb, pred_cb, pred_summer])
model_names = ['LightGBM','XGBoost','CatBoost','Summer-Specialist']

def blend_rmsle_harmonic(weights):
    w = np.maximum(weights, 0)
    if w.sum() < 1e-9: return 999.0
    return rmsle(y_raw[covered_mask], weighted_harmonic_mean(oof_stack[covered_mask], w))

result  = minimize(blend_rmsle_harmonic, x0=[0.4,0.2,0.2,0.2],
                   method='Nelder-Mead', options={'maxiter':3000,'xatol':1e-7})
weights = np.maximum(result.x, 0); weights = weights / weights.sum()

oof_blend  = weighted_harmonic_mean(oof_stack,  weights)
test_blend = weighted_harmonic_mean(test_stack, weights)
blend_score= rmsle(y_raw[covered_mask], oof_blend[covered_mask])

print('Ensemble weights:')
for name, w in zip(model_names, weights): print(f'  {name}: {w:.4f}')
print(f'\nOOF RMSLE:')
for name, scores in zip(model_names,[cv_scores_lgb,cv_scores_xgb,cv_scores_cb,cv_scores_summer]):
    print(f'  {name:<22}: {np.mean(scores):.5f}')
print(f'  {"Harmonic Ensemble":<22}: {blend_score:.5f}  <- target: lowest')
print(f'{elapsed()}')

## 9. Log-Bias Correction

In [ ]:
y_c   = y_raw[covered_mask]
oof_c = oof_blend[covered_mask]

multipliers     = np.arange(0.96, 1.01, 0.001)
mult_scores     = [rmsle(y_c, oof_c * m) for m in multipliers]
best_mult       = multipliers[np.argmin(mult_scores)]
best_mult_score = np.min(mult_scores)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(multipliers, mult_scores, color=PALETTE[0], lw=2)
ax.axvline(best_mult, color=PALETTE[2], ls='--', lw=1.5, label=f'Best={best_mult:.3f}')
ax.set_title(f'RMSLE vs Correction Multiplier  (best x{best_mult:.3f} -> {best_mult_score:.5f})')
ax.set_xlabel('Multiplier'); ax.set_ylabel('RMSLE'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Without correction: {rmsle(y_c, oof_c):.5f}')
print(f'With correction   : x{best_mult:.3f} -> {best_mult_score:.5f}')

## 10. Generate Submission

**YoY-anchored blend strategy:**  
For 42-day ahead forecasts on seasonal demand, blending model predictions with
same-period-last-year (vol_lag_364) anchors improves calibration. The YoY anchor
references June-July 2014 actuals — the same seasonal regime and hub operational
context as the test period. The model (20% weight) contributes daily variation
signals: promotional timing, weekday effects, and hub-specific trends.

`final = 0.80 * vol_lag_364 * yoy_growth + 0.20 * ensemble_model`

In [ ]:
import pandas as pd, numpy as np

test_raw  = pd.read_csv(DATA_PATH + 'orders_test.csv')
open_mask = test_raw['IsOpen'].values == 1

# ── Model predictions with log-bias correction ────────────────────────────
model_pred = np.maximum(test_blend * best_mult, 0)
model_pred[~open_mask] = 0.0

# ── YoY anchor: same day last year ───────────────────────────────────────
pred_yoy = X_test['vol_lag_364'].copy()
pred_yoy = pred_yoy.fillna(X_test['vol_lag_365']).fillna(X_test['vol_lag_371'])
pred_yoy = pred_yoy.values.copy()

# Apply measured YoY growth rate (year-over-year from training data)
_open_2015 = train_fe[(train_fe['IsOpen']==1) & (train_fe['Date'].dt.year==2015)]['OrderVolume'].mean()
_open_2014 = train_fe[(train_fe['IsOpen']==1) & (train_fe['Date'].dt.year==2014)]['OrderVolume'].mean()
yoy_growth  = _open_2015 / _open_2014 if _open_2014 > 0 else 1.013
pred_yoy    = pred_yoy * yoy_growth
pred_yoy[~open_mask] = 0.0
pred_yoy    = np.maximum(pred_yoy, 0)

print(f'YoY growth rate: {yoy_growth:.4f}')
print(f'Model pred open-day mean : {model_pred[open_mask].mean():.0f}')
print(f'YoY anchor open-day mean : {pred_yoy[open_mask].mean():.0f}')

# ── Final blend ──────────────────────────────────────────────────────────
YOY_WEIGHT  = 0.80
final_preds = YOY_WEIGHT * pred_yoy + (1 - YOY_WEIGHT) * model_pred
final_preds[~open_mask] = 0.0
final_preds = np.maximum(final_preds, 0)

print(f'\nFinal blend ({YOY_WEIGHT:.0%} YoY + {1-YOY_WEIGHT:.0%} model):')
print(f'  Open-day mean : {final_preds[open_mask].mean():.0f}')
print(f'  Overall mean  : {final_preds.mean():.0f}')
print(f'  Zeros (closed): {(final_preds==0).sum()} ({(final_preds==0).mean()*100:.1f}%)')

# ── Write submission ─────────────────────────────────────────────────────
submission = pd.DataFrame({
    'Id'          : test_raw['Id'],
    'OrderVolume' : np.round(final_preds, 2)
})
submission['OrderVolume'] = submission['OrderVolume'].clip(lower=0)
submission.to_csv('submission.csv', index=False)
print(f'\nsubmission.csv saved successfully.  {elapsed()}')
print(submission.head(10))

## 11. Final Summary

In [ ]:
rows = [
    ('LightGBM',             lgb_oof),
    ('XGBoost',              xgb_oof),
    ('CatBoost',             cb_oof),
    ('Summer Specialist',    summer_oof),
    ('Harmonic Ensemble',    blend_score),
    ('Ensemble + Correction',best_mult_score),
]
summary = pd.DataFrame(rows, columns=['Model','OOF RMSLE']).sort_values('OOF RMSLE')
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bar_cols = [PALETTE[1] if 'Correction' in m else (PALETTE[0] if 'Ensemble' in m else '#555')
            for m in summary['Model']]
bars = ax.bar(range(len(summary)), summary['OOF RMSLE'], color=bar_cols, alpha=0.87, width=0.6)
ax.set_xticks(range(len(summary)))
ax.set_xticklabels(summary['Model'], rotation=20, ha='right', fontsize=9)
ax.set_title('OOF RMSLE by Model (lower is better)', fontsize=12)
ax.set_ylabel('RMSLE')
ax.set_ylim(summary['OOF RMSLE'].min()*0.97, summary['OOF RMSLE'].max()*1.03)
ax.grid(True, axis='y', alpha=0.4)
for bar, val in zip(bars, summary['OOF RMSLE']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.0001, f'{val:.5f}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

total_min = (time.time()-GLOBAL_T0)/60
print(f'\nTotal runtime: {total_min:.1f} minutes')
print('submission.csv is ready.')